# 掩码数组与无效数据

学习目标：保持数组形状标记无效观测，计算有效数据的统计量，并在填充和转换时保留明确的缺失处理规则。

前置知识：数组、布尔掩码、缺失值、聚合、dtype、视图与副本。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章为按需选读专题。示例使用单元内的小数组，后续单元沿用首次导入的 np 和 ma。

## 1 保留位置并排除无效观测

两台设备各记录三次计数，-1 表示未采集到数据。希望计算每台设备的有效均值，同时保留原来的两行三列，可以使用 numpy.ma 的掩码数组。

ma 是 numpy.ma 的导入别名。masked_where 在条件为 True 的位置设置掩码，表示该观测无效；显示时用 -- 标记这些位置。掩码数组的 mean 跳过被屏蔽的位置，count 统计未屏蔽的元素个数。

In [1]:
import numpy as np
import numpy.ma as ma

counts = np.array([[10, -1, 14], [20, 22, -1]], dtype=np.int16)
valid_counts = ma.masked_where(counts < 0, counts)

print(valid_counts)  # [[10 -- 14], [20 22 --]]：保留原来的行列位置。
print(valid_counts.shape, valid_counts.dtype)  # (2, 3)，int16。
print(valid_counts.mean(axis=1))  # [12.0 21.0]：每台设备只用有效计数。
print(valid_counts.count(axis=1))  # [2 2]：两台设备各有两个有效观测。

[[10 -- 14]
 [20 22 --]]
(2, 3) int16
[12.0 21.0]
[2 2]


## 2 data 与 mask

MaskedArray 由底层数据和掩码共同描述。data 是数值数组，mask 表示哪些位置应排除；data 中仍可能保存无效记录的原值，不能脱离 mask 把它们当作有效观测。

ma.array 可以分别传入数据和掩码。下面显式用 copy=True 创建独立的数据副本，便于观察；默认 copy=False 可能复用输入数据。

| 属性或方法 | 中文名称／含义 |
| --- | --- |
| data | 底层数据，不包含有效性判断 |
| mask | 掩码，True 表示无效，False 表示未屏蔽 |
| count | 未屏蔽元素的数量 |
| getmaskarray | 取得与数据同形的布尔掩码 |

In [2]:
counts = np.array([5, -9, 7], dtype=np.int16)
observations = ma.array(counts, mask=[False, True, False], copy=True)

print(observations)  # [5 -- 7]
print(observations.data)  # [5 -9 7]：-9 仍在底层存储中。
print(observations.mask)  # [False True False]
print(observations.count())  # 2，不能用 size=3 代替有效计数。
print(observations.size, observations.dtype)  # 3，int16。

[5 -- 7]
[ 5 -9  7]
[False  True False]
2
3 int16


没有无效项时，mask 可能是标量 ma.nomask，显示为 False，而不是同形布尔数组。需要逐元素检查或组合掩码时，getmaskarray 总能给出与数据同形的布尔数组。

In [3]:
complete = ma.array([[1, 2, 3], [4, 5, 6]])
full_mask = ma.getmaskarray(complete)

print(complete.mask)  # False：本例未设置任何屏蔽位置。
print(full_mask)  # 两行三列，全部为 False。
print(full_mask.shape, full_mask.dtype)  # (2, 3)，bool。

False
[[False False False]
 [False False False]]
(2, 3) bool


## 3 按数据含义设置掩码

### 3.1 NaN 与无穷值

masked_invalid 屏蔽 NaN、正无穷和负无穷，并保留输入已有的掩码。它只判断数值是否有限，不知道某个有限值是否符合设备或业务规则。

下面的 999.0 虽然有限，但已被人工标记为无效；补充屏蔽 NaN 和无穷值时，应保留这个判断。

In [4]:
readings = ma.array(
    [20.0, np.nan, np.inf, 999.0], mask=[False, False, False, True],
)
cleaned = ma.masked_invalid(readings)

print(cleaned)  # [20.0 -- -- --]
print(ma.getmaskarray(cleaned))  # [False True True True]：原有掩码仍保留。
print(cleaned.shape, cleaned.dtype)  # (4,)，float64。
print(cleaned.count())  # 1

[20.0 -- -- --]
[False  True  True  True]
(4,) float64
1


### 3.2 任务规定的有效范围

若传感器有效范围约定为 0 到 50，有限的 99 也需要屏蔽。masked_where 可以表达这个条件，条件中的 True 表示要排除的位置。

In [5]:
readings = np.array([[18, 20, 99], [-1, 22, 24]], dtype=np.int16)
invalid = (readings < 0) | (readings > 50)
cleaned = ma.masked_where(invalid, readings)

print(cleaned)  # [[18 20 --], [-- 22 24]]
print(cleaned.count(axis=1))  # [2 2]
print(cleaned.shape, cleaned.dtype)  # (2, 3)，int16，无需先改为浮点数。

[[18 20 --]
 [-- 22 24]]
[2 2]
(2, 3) int16


## 4 与布尔筛选、NaN 的区别

### 4.1 筛出元素与保留形状

普通数组用同形布尔条件索引时，True 表示选中，结果是一维有效值序列。掩码数组的 mask 恰好相反：True 表示屏蔽，数组仍保留原来的形状。

只需要有效值列表时可以筛选；还要按设备或观测位置对齐时，掩码数组能保留这些位置。

In [6]:
counts = np.array([[10, -1, 14], [20, 22, -1]], dtype=np.int16)
valid = counts >= 0
selected = counts[valid]
masked = ma.array(counts, mask=~valid, copy=True)

print(selected)  # [10 14 20 22]
print(selected.shape, selected.dtype)  # (4,)，int16，原行列结构已压缩。
print(masked)  # 两行三列，两个 -1 位置显示为 --。
print(masked.shape, masked.dtype)  # (2, 3)，int16。

[10 14 20 22]
(4,) int16
[[10 -- 14]
 [20 22 --]]
(2, 3) int16


### 4.2 NaN 不是自动生成的掩码

NaN 是浮点数据中的特殊值；mask 是单独保存的有效性信息。直接把含 NaN 的数组交给 ma.array，不会自动把 NaN 位置设为屏蔽。count 只看 mask，所以要先明确屏蔽规则，再称它为有效计数。

np.nanmean 忽略 NaN，masked_invalid 则同时屏蔽 NaN 和无穷值。两种处理的排除条件不同。

In [7]:
readings = np.array([10.0, np.nan, np.inf])
plain_masked = ma.array(readings)
cleaned = ma.masked_invalid(readings)

print(plain_masked.mask, plain_masked.count())  # False、3：NaN 和 inf 尚未屏蔽。
print(np.nanmean(readings))  # inf：只忽略 NaN，正无穷仍参与计算。
print(cleaned)  # [10.0 -- --]
print(cleaned.mean(), cleaned.count())  # 10.0、1：只保留有限读数。

False 3
inf
[10.0 -- --]
10.0 1


## 5 填充与提取有效值

### 5.1 用 filled 返回普通数组

需要向只接受普通数组的接口传递数据时，可以用 filled 明确替换屏蔽位置。结果保持形状，但不再携带 mask；替换值只是编码约定，不意味着补出了真实观测。

下面用 -1 标记缺失位置，不修改原来的掩码数组。如果没有屏蔽元素，filled 可能复用底层数据，因此不要把它当作无条件复制方法。

In [8]:
counts = ma.array(
    [[10, 999, 14], [20, 22, 999]],
    mask=[[False, True, False], [False, False, True]], dtype=np.int16,
)
exported = counts.filled(-1)

print(exported)  # [[10 -1 14], [20 22 -1]]
print(type(exported).__name__, exported.shape, exported.dtype)  # ndarray，(2, 3)，int16。
print(counts)  # 原数组仍在两个位置显示 --。
print(counts.mean(axis=1))  # [12.0 21.0]：统计仍用原掩码。

[[10 -1 14]
 [20 22 -1]]
ndarray (2, 3) int16
[[10 -- 14]
 [20 22 --]]
[12.0 21.0]


### 5.2 填充值要适合 dtype

整数掩码数组可以保留整数类型，但不能直接把 NaN 填进整数数据。需要用 NaN 导出时，先把掩码数组转换成浮点类型，再调用 filled。

In [9]:
counts = ma.array([10, 999, 14], mask=[False, True, False], dtype=np.int16)

try:
    counts.filled(np.nan)
except TypeError as error:
    print(type(error).__name__, str(error))  # NaN 无法转换为 int16 填充值。
else:
    raise AssertionError("整数数组不能用 NaN 填充")

exported = counts.astype(np.float64).filled(np.nan)
print(exported)  # [10. nan 14.]
print(exported.shape, exported.dtype)  # (3,)，float64。
print(counts.dtype)  # int16：原掩码数组未变。

TypeError Cannot convert fill_value nan to dtype int16
[10. nan 14.]
(3,) float64
int16


### 5.3 用 compressed 提取有效值

compressed 返回所有未屏蔽元素组成的一维普通数组。它适合只需要数值序列的任务，不保留原行列位置。若全部元素被屏蔽，得到长度为 0 的一维数组。

In [10]:
counts = ma.array(
    [[10, -1, 14], [20, 22, -1]],
    mask=[[False, True, False], [False, False, True]], dtype=np.int16,
)
values = counts.compressed()
all_masked = ma.array([1, 2, 3], mask=True, dtype=np.int16)

print(values)  # [10 14 20 22]
print(type(values).__name__, values.shape, values.dtype)  # ndarray，(4,)，int16。
print(all_masked.compressed())  # []：没有未屏蔽元素。
print(all_masked.compressed().shape)  # (0,)

[10 14 20 22]
ndarray (4,) int16
[]
(0,)


## 6 运算中的掩码传播

### 6.1 两个输入共同决定有效性

同形掩码数组相减时，只要任一输入在某个位置被屏蔽，结果的对应位置也会被屏蔽。与标量运算时，已有掩码同样保留。

In [11]:
measured = ma.array([10, 20, 30], mask=[False, True, False])
baseline = ma.array([1, 2, 3], mask=[False, False, True])
difference = measured - baseline

print(difference)  # [9 -- --]：第二、第三个位置分别来自两个输入的掩码。
print(ma.getmaskarray(difference))  # [False True True]
print(difference + 1)  # [10 -- --]
print(difference.shape, difference.dtype)  # (3,)，本机为 int64。

[9 -- --]
[False  True  True]
[10 -- --]
(3,) int64


### 6.2 运算定义域之外的位置

ma 中的部分数值函数会屏蔽定义域之外的输入，例如实数平方根的负数输入。下面使用 ma.sqrt，负数位置和已有的屏蔽位置都不会成为有效结果。

被屏蔽位置的底层 data 不是可用的运算结果。不同操作还可能改变这些底层值，不能依赖它们始终保持原值，也不能用它们代替 mask 判断有效性。

In [12]:
values = ma.array([4.0, -1.0, 9.0], mask=[False, False, True])
roots = ma.sqrt(values)

print(roots)  # [2.0 -- --]：只有第一个位置是有效平方根。
print(ma.getmaskarray(roots))  # [False True True]
print(roots.data)  # 仅观察本次底层存储；后两个位置仍无效，不作为数值结论。
print(roots.count())  # 1

[2.0 -- --]
[False  True  True]
[ 2. -1.  9.]
1


## 7 聚合与有效计数

### 7.1 明确聚合轴

掩码数组的 sum 和 mean 按未屏蔽值聚合。axis=0 跨设备汇总每次观测，axis=1 汇总每台设备的多次观测；count 使用同样的轴约定。

keepdims=True 保留长度为 1 的聚合轴。下面每台设备的均值形状为 (2, 1)，可以广播回两行三列，计算有效位置的偏差。

In [13]:
counts = ma.array(
    [[10, -1, 14], [20, 22, -1]],
    mask=[[False, True, False], [False, False, True]], dtype=np.int16,
)
row_means = counts.mean(axis=1, keepdims=True)

print(counts.sum(axis=0))  # [30 22 14]
print(counts.count(axis=0))  # [2 1 1]：三次观测的有效设备数不同。
print(row_means, row_means.shape)  # [[12.0], [21.0]]，(2, 1)。
print(counts - row_means)  # [[-2.0 -- 2.0], [-1.0 1.0 --]]

[30 22 14]
[2 1 1]
[[12.0]
 [21.0]] (2, 1)
[[-2.0 -- 2.0]
 [-1.0 1.0 --]]


### 7.2 一整组都无效

某行全部被屏蔽时，这行的 count 为 0，sum 和 mean 的结果仍被屏蔽。不能把“没有有效观测”解释为“观测值为 0”；应同时查看聚合结果和有效计数。

In [14]:
counts = ma.array(
    [[10, 12, 14], [20, 22, 24]],
    mask=[[False, True, False], [True, True, True]], dtype=np.int16,
)
means = counts.mean(axis=1)

print(counts.count(axis=1))  # [2 0]
print(counts.sum(axis=1))  # [24 --]：第二行不是有效的零值。
print(means)  # [12.0 --]
print(ma.getmaskarray(means))  # [False True]

[2 0]
[24 --]
[12.0 --]
[False  True]


## 8 转换时检查掩码

np.asarray 返回普通 ndarray，会失去 MaskedArray 的掩码信息，而不是自动填充屏蔽位置。ma.asarray 则转换为掩码数组；输入已经携带的掩码可以保留。

如果先丢失 mask，再用 ma.asarray 包装普通数组，原来的有效性判断不会自动恢复。向其他接口传递数据时，应确认它支持掩码，或先用 filled 明确导出规则。

In [15]:
counts = ma.array([10, 99, 14], mask=[False, True, False], dtype=np.int16)
plain = np.asarray(counts)
preserved = ma.asarray(counts)
rewrapped = ma.asarray(plain)

print(plain)  # [10 99 14]：原来无效的 99 再次暴露为普通数值。
print(type(plain).__name__, plain.shape, plain.dtype)  # ndarray，(3,)，int16。
print(plain.mean(), counts.mean())  # 41.0、12.0：丢失掩码改变了统计含义。
print(ma.getmaskarray(preserved))  # [False True False]
print(ma.getmaskarray(rewrapped))  # [False False False]：无法从数值猜回原掩码。

[10 99 14]
ndarray (3,) int16
41.0 12.0
[False  True False]
[False False False]


## 本章小结

（1）掩码数组用 data 保存数值，用 mask 标记无效位置；mask=True 表示排除，与布尔索引的选中含义相反。

（2）masked_invalid 屏蔽非有限值，任务规定的无效范围需要另外表达；count 只按已有 mask 统计。

（3）filled 保持形状并返回普通数组，compressed 返回一维有效值；填充值必须适合 dtype，填充不等于恢复真实观测。

（4）掩码随运算传播，聚合要同时查看有效计数；全屏蔽组不能当作零值组，底层 data 不能替代有效结果。

（5）np.asarray 会丢失掩码，ma.asarray 不能从已经丢失信息的普通数组中恢复原掩码。

## 练习

（1）把负数计数屏蔽，保留三台设备、每台两次观测的结构。打印结果、shape、dtype、每台设备的有效均值和计数，单独识别完全没有有效观测的设备。

In [16]:
counts = np.array([[4, -1], [6, 8], [-1, -1]], dtype=np.int16)

# 在此创建掩码数组，按设备聚合并检查有效计数。
# 检查：形状为 (3, 2)，dtype 为 int16；最后一台设备的有效计数为 0。

（2）先预测以下三种处理的返回类型、形状，以及无效位置如何表示，再运行核对。解释为什么不能把三个结果直接当作含义相同的输入。

In [17]:
values = ma.array(
    [[1, 2, 3], [4, 5, 6]],
    mask=[[False, True, False], [True, False, False]], dtype=np.int16,
)
first = values.filled(-1)
second = values.compressed()
third = np.asarray(values)

# 先写预测，再打印类型、shape 和数值；不要只比较元素总数。
print(type(first).__name__, first.shape, first)
print(type(second).__name__, second.shape, second)
print(type(third).__name__, third.shape, third)

ndarray (2, 3) [[ 1 -1  3]
 [-1  5  6]]
ndarray (4,) [1 3 5 6]
ndarray (2, 3) [[1 2 3]
 [4 5 6]]


（3）最初只需得到所有有效读数的一维列表；后来增加要求：保留原来的设备与观测位置，并计算每行有效均值。请为两个要求分别选择处理方法，用注释解释理由。有效范围为 0 到 50，并且要求数值有限。

In [18]:
readings = np.array([[18.0, np.nan, 99.0], [20.0, np.inf, 24.0]])

# 在此组合非有限与范围条件，分别完成“只取有效值”和“保留原形状”。
# 检查：有效数值共三个；保形结果为 (2, 3)，两行有效计数分别为 1、2。
# 在此说明为什么压缩后的一维结果不再保留原行列对应关系。

（4）外部接口只接收普通浮点数组，并约定 NaN 表示缺失。请按这个约定导出下面的整数掩码数组。解释为什么不能直接使用 np.asarray，也不能直接对整数数组填 NaN。

In [19]:
counts = ma.array([5, 999, 9], mask=[False, True, False], dtype=np.int16)

# 在此转换为浮点掩码数组后填 NaN，打印结果、shape、dtype 与 NaN 位置。
# 检查：原数组仍是 int16，原掩码不变；导出结果的第二个位置为 NaN。
# 在此用注释说明两个直接转换方案的问题。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（2.5） | [The numpy.ma module](https://numpy.org/doc/2.5/reference/maskedarray.generic.html)：What is a masked array、Accessing the data/mask、Operations on masked arrays，尤其底层无效数据可能被操作改变的 Warning；[MaskedArray 与常量](https://numpy.org/doc/2.5/reference/maskedarray.baseclass.html)：data、mask、nomask、-- 显示；[ma.array](https://numpy.org/doc/2.5/reference/generated/numpy.ma.array.html)：mask、dtype、copy、keep_mask；[masked_where](https://numpy.org/doc/2.5/reference/generated/numpy.ma.masked_where.html)、[masked_invalid](https://numpy.org/doc/2.5/reference/generated/numpy.ma.masked_invalid.html)：屏蔽条件及原掩码保留；[getmaskarray](https://numpy.org/doc/2.5/reference/generated/numpy.ma.getmaskarray.html)：同形布尔掩码；[filled](https://numpy.org/doc/2.5/reference/generated/numpy.ma.MaskedArray.filled.html)、[compressed](https://numpy.org/doc/2.5/reference/generated/numpy.ma.MaskedArray.compressed.html)：返回类型、形状及复制条件；[mean](https://numpy.org/doc/2.5/reference/generated/numpy.ma.MaskedArray.mean.html)、[sum](https://numpy.org/doc/2.5/reference/generated/numpy.ma.MaskedArray.sum.html)、[count](https://numpy.org/doc/2.5/reference/generated/numpy.ma.MaskedArray.count.html)：屏蔽项、axis 与 keepdims；[nanmean](https://numpy.org/doc/2.5/reference/generated/numpy.nanmean.html)：只忽略 NaN；[asarray](https://numpy.org/doc/2.5/reference/generated/numpy.asarray.html)、[ma.asarray](https://numpy.org/doc/2.5/reference/generated/numpy.ma.asarray.html)：普通数组与掩码数组的转换；[Boolean array indexing](https://numpy.org/doc/2.5/user/basics.indexing.html#boolean-array-indexing)：同形布尔索引的选中语义与一维结果。 |
| NumPy 官方源码（GitHub，v2.5.0） | [numpy/ma/core.py](https://github.com/numpy/numpy/blob/v2.5.0/numpy/ma/core.py)：_check_fill_value（467–517 行）与 filled（3857–3936 行）的填充值类型检查；sum（5198–5259 行）、mean（5376–5430 行）的掩码与有效计数处理。 |